# Web Analytics - Assignment 2 (Part-1)

*June 15, 2017*

## Group 1 Members:
* Mauricio Alarcon
* Sekhar Mekala
* Aadi Kalloo
* Srinivasa Illapani
* Param Singh

**The requirements of this assignment are given below:**

Load a graph database of your choosing from a text file or other source.  If you take a large network dataset from the web (such as fromhttps://snap.stanford.edu/data/), please feel free at this point to load just a small subset of the nodes and edges.

Create basic analysis on the graph, including the graph?s diameter, and at least one other metric of your choosing.  You may either code the functions by hand (to build your intuition and insight), or use functions in an existing package. 

Use a visualization tool of your choice (Neo4j, Gephi, etc.) to display information.
Please record a short video (~ 5 minutes), and submit a link to the video in advance of our meet-up.

### Approach

In this assignment, we will be creating a graph based on the Super Heros data set available at https://www.kaggle.com/csanhueza/the-marvel-universe-social-network. We will identify the diameter of the graph and the radius of the graph, along with the graph's center nodes. 

Importing all the required packages.

In [ ]:
import networkx as net
import urllib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sklearn

## Data
We downloaded the following 3 datasets from  https://www.kaggle.com/csanhueza/the-marvel-universe-social-network.

* nodes.csv - Contains comma separated values of character and the character type. The type of the character can be either *hero* or *comic*.

* edges.csv - Contains comma separated values of the hero's name, and the comic book name.

* hero-network.csv - Contains comma separated values of the hero1 and hero2 who appeared together in a comic.

In this assignment we will be using the file *hero-network.csv* to build the graph of super heroes network based on the Marvel comics.

We will build an undirected graph using the data present in *hero-network.csv* file. It has 2 columns *hero1* and *hero2*. The data in this file represents the superheroes who appeared together in a comic book.

### Reading and cleansing the data
We will read the data into a data frame and perform some sanity checks to make sure that the data is suitable for building a graph.

_Sample records of the hero-network.csv are displayed below:_

In [ ]:
#Read the data into a pandas data frame:
ga_edges = pd.read_csv('./data/hero-network.csv')

#print("Sample rows...")
display(ga_edges.head())


_Summary information of the data set is displayed below:_

In [ ]:

#print("Summary information of the data is displayed below:")
display(ga_edges.describe())

#print("\nThe above summary confirms that there are no NA values in the data set, \
#\nand CAPTAIN AMERICA has co-appeared most of the times along with other comic characters.")


The above summary confirms that there are no NA values in the data set, 
and CAPTAIN AMERICA has co-appeared most of the times along with other comic characters.

_Do we have any self referential nodes?_

We will eliminate self-referential records (if any) present in the data set, since we are not interested in analyzing such nodes in this project.

In [ ]:



print("There are {0} rows which have both \
the same hero listed as hero1 and hero2 in the same comic\n\
Some of these rows are displayed below:".
      format(sum(ga_edges["hero1"] == ga_edges["hero2"])))


display(ga_edges.loc[(ga_edges["hero1"] == ga_edges["hero2"])].head())



There are 2232 rows which are self referencing. We will eliminate these rows from the data frame, and build the graph using the remaining rows.

In [ ]:
#Deleting the self referential endges from the data frame
ga_edges = ga_edges.loc[~(ga_edges["hero1"] == ga_edges["hero2"])]

_Summary information of the final data frame, along with some sample records are displayed below:_

In [ ]:
print("\nThe summary information of the final data frame:")

display(ga_edges.describe())
print("Sample records of the final, cleansed data set are displayed below:")
display(ga_edges.head())


### Building the graph
Using the cleansed data, we will build a network graph of super heros.

In [ ]:
#Building a graph directly using pandas data frame:
try:
    GA = net.from_pandas_dataframe(ga_edges,source="hero1",target="hero2")
    print("Graph built sucessfully")
except:
    print("Error in building the graph")


## Analysis

The summary information of the graph built is given below:

In [ ]:
print(net.info(GA))

The above summary confirms that our graph has 6426 nodes and 167207 edges. The average degree of nodes is 52.0408. This means, each node has approximately 52 connections with other nodes. This also signifies that each comic character has paired up with other comic characters for 52 times, on an average.

Before we discuss about the metrics of interest, let us define some terms:

_Eccentricity_ The eccentricity of a node $n$ in a graph represents the _maximum_ of the shortest path distances from $n$ to all other graphs

_Diameter_ The diameter of a graph is the _maximum_ eccentricity of all the nodes in the graph

_Radius_ The radius of a graph is the _minimum_ eccentricity of all the nodes in the graph

_Graph center_ The set of all nodes whose eccentricity is equal to the graph's radius

_Graph periphery_ The set of all nodes whose eccentricity is equal to the diameter of the graph

In this assignment we will obtain the following metrics of the graph:

* Diameter
* Radius
* Graph Center
* Graph Periphery



_Let us identify if we have any disconnected components in our graph_
If we have any disconnected components in our graph, we will separate them into subgraphs.

In [ ]:
net.is_connected(GA)

The above block's output confirms that we have some disconnected components. Let us identify the disconnected components and collect them into a list of sub-graphs

In [ ]:
GA_list = list(net.connected_component_subgraphs(GA))
print("There are {0} subgraphs".format(len(GA_list)))

x = 1
for i in GA_list:
    print("----------------------------------")
    print("Summary details of sub-graph: {0} ".format(x))
    print("----------------------------------")
    x += 1
    print(net.info(i))
    print("Diameter:{0}".format(net.diameter(i)))
    print("Radius:{0}".format(net.radius(i)))

## Subgraph-1 analysis
The above display shows that the subgraph-1 has a radius of 3 and a diameter of 5. This means, the maximum shortest distance from some nodes to all other nodes is 5 hops and the minimum shortest distance is 3 hops. The subgraph-1 is also the biggest graph, since it has 6408 nodes, with 167151 edges, and the average connections per node is approximately 52. Let us analyze how the eccentricity values are distributed.

In [ ]:
eccentricities = net.eccentricity(GA_list[0])

In [ ]:
#Create a Pandas data frame for easy computation
ecc_sub_graph_df = pd.DataFrame(list(eccentricities.items()))

#Rename the pandas data frame columns
ecc_sub_graph_df.columns = ["Hero","Eccentricity"] 

#Display a set of rows
#display(ecc_sub_graph_df.head())

#Get the eccentricities counts
ecc_group = ecc_sub_graph_df.groupby(["Eccentricity"]).count().reset_index()
ecc_group.columns=["Eccentricity","Count"]

#Plot a barchart of eccentricity counts
N = 3
ind = np.arange(N)
width = .3
fig,ax=plt.subplots()
rects1 = ax.bar(ind, ecc_group["Count"], width, color='r')
ax.set_ylim([0,7000])
ax.set_ylabel('Counts')
ax.set_xlabel('Eccentricity')
ax.set_xticks(ind)
ax.set_xticklabels(('3','4','5'))

def autolabel(rects):
    """
    Attach a text label above each bar displaying its height
    """
    for rect in rects:
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., 1.05*height,
                '%d' % int(height),
                ha='center', va='bottom')

autolabel(rects1)

plt.show()

The above figure shows that only 296 nodes have the same eccentricity values as the radius of the subgraph-1 (radius of subgraph-1 is 3), while 212 nodes have eccentricity values equal to the graph's diameter. So based on these measures, we have 296 nodes as the center nodes, and 212 nodes as the periphery nodes. We will plot a graph with only the center 296 nodes and which have a degree range of 290-295. This will help us to reduce the number of nodes significantly and display some of the center nodes which have 290 to 295 connections (we found using Gephi that the center nodes have degree range of 52 to 295). To use Gephi, we built a new graph with just the center nodes. The following code will build a Networkx graph with just the center nodes.

In [ ]:
Hero = list(ecc_sub_graph_df[ecc_sub_graph_df["Eccentricity"] == 3]["Hero"])
sub_graph_0_simple=ga_edges[(ga_edges["hero1"].isin(Hero) & ga_edges["hero2"].isin(Hero))]

try:
    GA_0 = net.from_pandas_dataframe(sub_graph_0_simple,source="hero1",target="hero2")
    print("Graph built sucessfully")
except:
    print("Error in building the graph")

#Saving the graph
net.write_gml(GA_0,"subgraph0.gml")
print("Center nodes only graph summary:")
print(net.info(GA_0))

We can see that the center node graph has an average degree of 150, and given that we have 296 nodes in the graph, we can conclude that the graph is highly connected. The following graph shows the nodes with a degree range of 290 to 295.

<img src="images/subgraph0.png" style="max-width:100%; width: 50%">

The above graph confirms that the famous super heroes such as Sprder Man, Iron Man, Captain America, Hulk and Wolverine are very important Marvel characters, which are at the center of the graph with maximu number of connections to other characters.


### Sub-graph 2, 3 and 4 analysis
The summary details of the sub-graphs 2, 3 and 4 show that these graphs are ver small with just 7, 9, and 2 nodes respectively. These graphs are plotted below:



In [ ]:
#Define a function to plot graphs in Networkx
def draw_netx_graph(g):
    layout = net.spring_layout(g)
    net.draw_networkx_nodes(g,layout)
    net.draw_networkx_labels(g,pos=layout)
    net.draw_networkx_edges(g,pos=layout)
    plt.axis("off")
    plt.show()


**Subgraph-2**

In [ ]:
draw_netx_graph(GA_list[1])

**Subgraph-3**

In [ ]:
draw_netx_graph(GA_list[2])

**Subgraph-4**

In [ ]:
draw_netx_graph(GA_list[3])

The subgraphs 2, 3, and 4 are very small, and the characters in these graphs are not isolated, and perhaps they were not used in more than one comic.

## Conclusion
In this assignment we were able to identify the following:

* The Super heroes network data set has 4 isolated graphs.
* Except one isolated graph, the remaining isolated graphs are not big enough to draw any interesting conclusions, other than the fact that those comic characters have never co-occured with the famous comic characters.
* We were able to identify the diameter, radius, center and peripheri of the biggest isolated graph.